# 03 — Feature engineering

Builds the feature table used for training. All features are **leak-free** — they use only past matches relative to each row.

**Features**: pre-match Elo (home/away), rolling form (last 5 & 10), head-to-head record (last 5), tournament importance, neutral-ground flag, days since each team's last match.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.elo import EloEngine, k_for
from src.features import rolling_form, h2h_features, label_result
from src.wc2026_config import normalize

df = pd.read_csv(ROOT / 'data' / 'raw' / 'results.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_team'] = df.home_team.map(normalize)
df['away_team'] = df.away_team.map(normalize)
# Drop unplayed fixtures (e.g. the WC2026 schedule that ships in this CSV).
unplayed = df[df.home_score.isna() | df.away_score.isna()]
print('unplayed fixtures dropped:', len(unplayed))
if len(unplayed):
    unplayed.to_csv(ROOT / 'data' / 'processed' / 'unplayed_fixtures.csv', index=False)
df = df.dropna(subset=['home_score','away_score']).reset_index(drop=True)
print(len(df), 'played matches')


In [ ]:
# --- Pre-match Elo (online update, walk-forward, no leakage) ---
engine = EloEngine()
h_elo_pre, a_elo_pre = [], []
for _, row in df.iterrows():
    h_elo_pre.append(engine.get(row.home_team))
    a_elo_pre.append(engine.get(row.away_team))
    engine.update(row.home_team, row.away_team,
                  int(row.home_score), int(row.away_score),
                  tournament=row.tournament, neutral=bool(row.neutral))
df['home_elo'] = h_elo_pre
df['away_elo'] = a_elo_pre
df['elo_diff'] = df.home_elo - df.away_elo
print('Elo computed. sample:')
df[['date','home_team','away_team','home_elo','away_elo']].tail()


In [ ]:
# Persist final Elo table (used by the WC2026 prediction notebook).
final = pd.DataFrame({'team': list(engine.ratings.keys()), 'elo': list(engine.ratings.values())})
final = final.sort_values('elo', ascending=False)
(ROOT / 'data' / 'processed').mkdir(parents=True, exist_ok=True)
final.to_csv(ROOT / 'data' / 'processed' / 'final_elo.csv', index=False)
final.head(20)


In [ ]:
# --- Rolling form (last 5 and last 10) ---
df = rolling_form(df, window=5).rename(columns={
    'home_form_pts':'home_form5_pts','home_form_gf':'home_form5_gf','home_form_ga':'home_form5_ga',
    'away_form_pts':'away_form5_pts','away_form_gf':'away_form5_gf','away_form_ga':'away_form5_ga',
})
df = rolling_form(df, window=10).rename(columns={
    'home_form_pts':'home_form10_pts','home_form_gf':'home_form10_gf','home_form_ga':'home_form10_ga',
    'away_form_pts':'away_form10_pts','away_form_gf':'away_form10_gf','away_form_ga':'away_form10_ga',
})
print('form done')


In [ ]:
# --- Head-to-head (last 5 meetings) ---
df = h2h_features(df, window=5)
print('h2h done')


In [ ]:
# --- Other features ---
df['tournament_k'] = df.tournament.map(k_for)
df['neutral_int'] = df.neutral.astype(int)
df['year'] = df.date.dt.year

# Days since each team's last match (rest)
long = pd.concat([
    df[['date','home_team']].rename(columns={'home_team':'team'}).assign(idx=df.index, side='h'),
    df[['date','away_team']].rename(columns={'away_team':'team'}).assign(idx=df.index, side='a'),
]).sort_values(['team','date'])
long['rest'] = long.groupby('team')['date'].diff().dt.days.fillna(365).clip(upper=365)
rest_h = long[long.side=='h'].set_index('idx')['rest']
rest_a = long[long.side=='a'].set_index('idx')['rest']
df['home_rest'] = rest_h
df['away_rest'] = rest_a

df['target'] = df.apply(label_result, axis=1)
df['goal_diff'] = df.home_score - df.away_score
print('total features done, shape:', df.shape)


In [ ]:
out = ROOT / 'data' / 'processed' / 'matches_features.parquet'
df.to_parquet(out, index=False)
print('wrote', out, df.shape)
df.head()
